# Results Outcome (Custom Analysis)

This notebook contains five analysis blocks:

1. Daily feature evolution plots (one plot per feature) plus descriptive statistics
2. Loss delta plots by feature combination and metric
3. Mean and standard deviation of loss across feature variations
4. QLIKE delta tables in LaTeX format
5. Correlation heatmaps between the daily variables and $\sqrt{RV}$

Configured feature combinations:
- None
- ratio_event
- exceedance_down
- lambda_t
- [ratio_event, exceedance_down]
- [exceedance_down, lambda_t]
- [ratio_event, exceedance_down, lambda_t]

In [47]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as px

# ---------- User controls ----------
SELECTED_COINS = None  # e.g. ["BTC", "ETH"] or None for all coins found
METRICS_FOR_DELTA = ["mse", "mae", "qlike"]
MODEL_FAMILY_FILTER = None   # e.g. ["HAR", "LSTM"] or None for all

FEATURE_COLUMNS = ["RV", "ratio_event", "exceedance_down", "lambda_t"]
FEATURE_COMBINATIONS = [
    None,
    "ratio_event",
    "exceedance_down",
    "lambda_t",
    ["ratio_event", "exceedance_down"],
    ["exceedance_down", "lambda_t"],
    ["ratio_event", "exceedance_down", "lambda_t"],
]


def combo_to_label(combo):
    if combo is None:
        return "None"
    if isinstance(combo, list):
        return "[" + ", ".join(combo) + "]"
    return str(combo)


COMBO_ORDER = [combo_to_label(c) for c in FEATURE_COMBINATIONS]

root = Path.cwd()
if not (root / "results").exists() and (root.parent / "results").exists():
    root = root.parent

results_root = root / "results"
daily_rv_root = root / "data" / "interim" / "daily_rv"

if not results_root.exists():
    raise FileNotFoundError(f"Results directory not found: {results_root}")
if not daily_rv_root.exists():
    raise FileNotFoundError(f"Daily RV directory not found: {daily_rv_root}")

result_coin_dirs_all = sorted(p for p in results_root.glob("coin=*") if p.is_dir())
data_coin_files_all = sorted(daily_rv_root.glob("*_rv.parquet"))

if not result_coin_dirs_all:
    raise FileNotFoundError(f"No coin directories found under: {results_root}")
if not data_coin_files_all:
    raise FileNotFoundError(f"No daily RV parquet files found under: {daily_rv_root}")

available_result_coins = sorted(p.name.replace("coin=", "") for p in result_coin_dirs_all)
available_data_coins = sorted(fp.stem.replace("_rv", "") for fp in data_coin_files_all)

if SELECTED_COINS is None:
    RESULT_COINS = available_result_coins
    DATA_COINS = available_data_coins
else:
    selected = {str(c).upper() for c in SELECTED_COINS}
    RESULT_COINS = [coin for coin in available_result_coins if coin.upper() in selected]
    DATA_COINS = [coin for coin in available_data_coins if coin.upper() in selected]

if not RESULT_COINS:
    raise ValueError("No matching result coins found for SELECTED_COINS")
if not DATA_COINS:
    raise ValueError("No matching daily-data coins found for SELECTED_COINS")

coin_dirs = [p for p in result_coin_dirs_all if p.name.replace("coin=", "") in RESULT_COINS]

print(f"Using results root: {results_root}")
print(f"Using daily RV root: {daily_rv_root}")
print(f"Result coins ({len(RESULT_COINS)}): {RESULT_COINS}")
print(f"Data coins ({len(DATA_COINS)}): {DATA_COINS}")
print(f"Metrics for delta plot: {METRICS_FOR_DELTA}")

Using results root: c:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results
Using daily RV root: c:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv
Result coins (2): ['BTC', 'ETH']
Data coins (2): ['BTC', 'ETH']
Metrics for delta plot: ['mse', 'mae', 'qlike']


In [48]:
# 1) Daily feature evolution + descriptive statistics

feature_frames = []

for coin in DATA_COINS:
    rv_path = daily_rv_root / f"{coin}_rv.parquet"
    if not rv_path.exists():
        print(f"Skipping {coin}: daily file not found -> {rv_path}")
        continue

    df = pd.read_parquet(rv_path).copy()

    if isinstance(df.index, pd.DatetimeIndex):
        df["date"] = pd.to_datetime(df.index, errors="coerce")
    elif "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    elif "time" in df.columns:
        df["date"] = pd.to_datetime(df["time"], errors="coerce")
    else:
        df["date"] = pd.NaT

    df["coin"] = coin
    available = [c for c in FEATURE_COLUMNS if c in df.columns]
    if not available:
        print(f"Skipping {coin}: none of {FEATURE_COLUMNS} found")
        continue

    keep_cols = ["date", "coin"] + available
    feature_frames.append(df[keep_cols])

if not feature_frames:
    raise RuntimeError("No daily feature data found for selected coins")

features_df = pd.concat(feature_frames, ignore_index=True)
features_df = features_df.dropna(subset=["date"]).copy()

for col in FEATURE_COLUMNS:
    if col in features_df.columns:
        features_df[col] = pd.to_numeric(features_df[col], errors="coerce")

for coin in sorted(features_df["coin"].dropna().astype(str).unique().tolist()):
    coin_df = features_df.loc[features_df["coin"].astype(str) == coin].copy()

    for feature in FEATURE_COLUMNS:
        if feature not in coin_df.columns:
            print(f"Skipping {coin} | {feature}: column not present")
            continue

        plot_df = coin_df[["date", feature]].dropna(subset=[feature]).copy()
        if plot_df.empty:
            print(f"Skipping {coin} | {feature}: no non-null numeric values")
            continue

        plot_df = plot_df.sort_values("date").reset_index(drop=True)

        fig = px.line(
            plot_df,
            x="date",
            y=feature,
            title=f"{coin} | Daily evolution of {feature}",
            labels={"date": "Date", feature: feature},
        )
        fig.update_layout(template="plotly_white")
        fig.show()

        stats_table = pd.DataFrame(
            {
                "coin": [coin],
                "feature": [feature],
                "min": [plot_df[feature].min()],
                "q25": [plot_df[feature].quantile(0.25)],
                "median": [plot_df[feature].median()],
                "q75": [plot_df[feature].quantile(0.75)],
                "max": [plot_df[feature].max()],
                "mean": [plot_df[feature].mean()],
                "std": [plot_df[feature].std()],
            }
        )

        print(f"Descriptive statistics for {coin} | {feature}")
        display(stats_table)

Descriptive statistics for BTC | RV


,coin,feature,min,q25,median,q75,max,mean,std
0,BTC,RV,0.000008,0.000346,0.000696,0.001507,0.111207,0.001677,0.004245


Descriptive statistics for BTC | ratio_event


,coin,feature,min,q25,median,q75,max,mean,std
0,BTC,ratio_event,0.0,0.027778,0.048611,0.069444,0.253472,0.052428,0.032399


Descriptive statistics for BTC | exceedance_down


...

,coin,feature,min,q25,median,q75,max,mean,std
0,ETH,ratio_event,0.0,0.027778,0.045139,0.069444,0.229167,0.052557,0.032944


Descriptive statistics for ETH | exceedance_down


,coin,feature,min,q25,median,q75,max,mean,std
0,ETH,exceedance_down,0.0,0.00392,0.007119,0.012375,0.182094,0.010162,0.011127


Descriptive statistics for ETH | lambda_t


,coin,feature,min,q25,median,q75,max,mean,std
0,ETH,lambda_t,0.405448,0.601413,0.64794,0.699899,0.882008,0.651083,0.074506


## 2) Loss Delta vs Baseline by Feature Combination

This chart uses:
- x-axis: selected feature combinations in fixed order
- y-axis: feature_model_loss - baseline_model_loss

Baseline is the `None` combination for the same coin and model.

In [49]:
def extract_feature_tag(path: Path) -> str:
    match = re.search(r"features=([^.]*)", path.name)
    return match.group(1) if match else "unknown"


def extract_model_name(path: Path) -> str:
    match = re.search(r"model=([^_]*)", path.name)
    return match.group(1) if match else "unknown_model"


def parse_feature_set(feature_tag: str) -> set[str]:
    tag = str(feature_tag)

    if tag == "base":
        return set()

    if tag.startswith("base+"):
        raw_parts = [p for p in tag.replace("base+", "").split("+") if p]
    else:
        raw_parts = []
        if "ratio_event" in tag or "perc_event_down" in tag:
            raw_parts.append("ratio_event")
        if "exceedance_down" in tag:
            raw_parts.append("exceedance_down")
        if "lambda_t" in tag or "lambda_hawkes" in tag or "lambda_daily" in tag:
            raw_parts.append("lambda_t")

    normalized = []
    for p in raw_parts:
        if p == "perc_event_down":
            normalized.append("ratio_event")
        elif p in {"lambda_hawkes", "lambda_daily"}:
            normalized.append("lambda_t")
        else:
            normalized.append(p)

    return set(normalized)


def feature_tag_to_combo_label(feature_tag: str):
    feature_set = parse_feature_set(feature_tag)

    mapping = {
        frozenset(): "None",
        frozenset({"ratio_event"}): "ratio_event",
        frozenset({"exceedance_down"}): "exceedance_down",
        frozenset({"lambda_t"}): "lambda_t",
        frozenset({"ratio_event", "exceedance_down"}): "[ratio_event, exceedance_down]",
        frozenset({"exceedance_down", "lambda_t"}): "[exceedance_down, lambda_t]",
        frozenset({"ratio_event", "exceedance_down", "lambda_t"}): "[ratio_event, exceedance_down, lambda_t]",
    }

    return mapping.get(frozenset(feature_set), None)


summary_frames = []
for coin_dir in coin_dirs:
    coin = coin_dir.name.replace("coin=", "")
    for fp in sorted(coin_dir.glob("summary__*.csv")):
        s = pd.read_csv(fp)

        if "feature_tag" not in s.columns:
            s["feature_tag"] = extract_feature_tag(fp)
        if "model_name" not in s.columns:
            s["model_name"] = extract_model_name(fp)
        if "model_kind" not in s.columns:
            s["model_kind"] = s["model_name"].astype(str).str.split("-", n=1).str[0].str.upper()

        s["coin"] = coin
        s["combo_label"] = s["feature_tag"].map(feature_tag_to_combo_label)
        summary_frames.append(s)

if not summary_frames:
    raise FileNotFoundError("No summary files found in results/coin=* directories")

summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.loc[summary_df["combo_label"].isin(COMBO_ORDER)].copy()

if MODEL_FAMILY_FILTER is not None:
    fam_set = {str(v).upper() for v in MODEL_FAMILY_FILTER}
    summary_df = summary_df.loc[summary_df["model_kind"].astype(str).str.upper().isin(fam_set)].copy()

available_metrics = [m for m in METRICS_FOR_DELTA if m in summary_df.columns]
missing_metrics = [m for m in METRICS_FOR_DELTA if m not in summary_df.columns]
if missing_metrics:
    print(f"Skipping missing metrics: {missing_metrics}")
if not available_metrics:
    raise ValueError("None of METRICS_FOR_DELTA are available in summary columns")

for metric in available_metrics:
    metric_df = summary_df.copy()

    for coin in sorted(metric_df["coin"].dropna().astype(str).unique().tolist()):
        coin_metric_df = metric_df.loc[metric_df["coin"].astype(str) == coin].copy()

        agg = (
            coin_metric_df.groupby(["model_kind", "model_name", "combo_label"], as_index=False)[metric]
            .mean(numeric_only=True)
        )

        baseline = (
            agg.loc[agg["combo_label"] == "None", ["model_kind", "model_name", metric]]
            .rename(columns={metric: "baseline_loss"})
        )

        delta_df = agg.merge(baseline, on=["model_kind", "model_name"], how="left")
        delta_df = delta_df.dropna(subset=["baseline_loss"]).copy()
        delta_df["coin"] = coin
        delta_df["loss_delta_vs_baseline"] = delta_df[metric] - delta_df["baseline_loss"]
        delta_df["combo_label"] = pd.Categorical(delta_df["combo_label"], categories=COMBO_ORDER, ordered=True)
        delta_df = delta_df.sort_values(["model_kind", "model_name", "combo_label"]).reset_index(drop=True)

        print(f"\nCoin: {coin} | Metric: {metric.upper()} | Rows in delta table: {len(delta_df)}")
        display(
            delta_df[
                [
                    "coin",
                    "model_kind",
                    "model_name",
                    "combo_label",
                    metric,
                    "baseline_loss",
                    "loss_delta_vs_baseline",
                ]
            ]
        )

        fig = px.line(
            delta_df,
            x="combo_label",
            y="loss_delta_vs_baseline",
            color="model_name",
            line_dash="model_kind",
            markers=True,
            category_orders={"combo_label": COMBO_ORDER},
            title=f"{coin} | Loss delta vs baseline by feature combination ({metric.upper()})",
            labels={
                "combo_label": "Feature combination",
                "loss_delta_vs_baseline": f"{metric} - baseline {metric}",
                "model_name": "Model variant",
                "model_kind": "Family",
            },
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(template="plotly_white")
        fig.show()


Coin: BTC | Metric: MSE | Rows in delta table: 21


,coin,model_kind,model_name,combo_label,mse,baseline_loss,loss_delta_vs_baseline
0,BTC,HAR,HAR-baseline,None,0.000111,0.000111,0.000000e+00
1,BTC,HAR,HAR-baseline,ratio_event,0.000109,0.000111,-2.085161e-06
2,BTC,HAR,HAR-baseline,exceedance_down,0.000111,0.000111,1.361571e-08
3,BTC,HAR,HAR-baseline,lambda_t,0.000109,0.000111,-1.648116e-06
4,BTC,HAR,HAR-baseline,"[ratio_event, exceedance_down]",0.000109,0.000111,-1.746653e-06
5,BTC,HAR,HAR-baseline,"[exceedance_down, lambda_t]",0.000109,0.000111,-1.529769e-06
6,BTC,HAR,HAR-baseline,"[ratio_event, exceedance_down, lambda_t]",0.000107,0.000111,-3.433718e-06
7,BTC,LSTM,LSTM-hs32,None,0.000147,0.000147,0.000000e+00
8,BTC,LSTM,LSTM-hs32,ratio_event,0.000153,0.000147,5.868016e-06
9,BTC,LSTM,LSTM-hs32,exceedance_down,0.000151,0.000147,3.614902e-06



Coin: ETH | Metric: MSE | Rows in delta table: 21


,coin,model_kind,model_name,combo_label,mse,baseline_loss,loss_delta_vs_baseline
0,ETH,HAR,HAR-baseline,None,0.000167,0.000167,0.000000e+00
1,ETH,HAR,HAR-baseline,ratio_event,0.000164,0.000167,-2.529972e-06
2,ETH,HAR,HAR-baseline,exceedance_down,0.000163,0.000167,-3.154965e-06
3,ETH,HAR,HAR-baseline,lambda_t,0.000168,0.000167,1.156057e-06
4,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down]",0.000163,0.000167,-3.649129e-06
5,ETH,HAR,HAR-baseline,"[exceedance_down, lambda_t]",0.000167,0.000167,-4.880675e-08
6,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down, lambda_t]",0.000166,0.000167,-5.414399e-07
7,ETH,LSTM,LSTM-hs32,None,0.000237,0.000237,0.000000e+00
8,ETH,LSTM,LSTM-hs32,ratio_event,0.000234,0.000237,-3.101137e-06
9,ETH,LSTM,LSTM-hs32,exceedance_down,0.000252,0.000237,1.483040e-05



Coin: BTC | Metric: MAE | Rows in delta table: 21


...

,coin,model_kind,model_name,combo_label,mae,baseline_loss,loss_delta_vs_baseline
0,ETH,HAR,HAR-baseline,None,0.008807,0.008807,0.000000
1,ETH,HAR,HAR-baseline,ratio_event,0.008733,0.008807,-0.000074
2,ETH,HAR,HAR-baseline,exceedance_down,0.008740,0.008807,-0.000067
3,ETH,HAR,HAR-baseline,lambda_t,0.008902,0.008807,0.000095
4,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down]",0.008751,0.008807,-0.000056
5,ETH,HAR,HAR-baseline,"[exceedance_down, lambda_t]",0.008939,0.008807,0.000133
6,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down, lambda_t]",0.008943,0.008807,0.000136
7,ETH,LSTM,LSTM-hs32,None,0.010811,0.010811,0.000000
8,ETH,LSTM,LSTM-hs32,ratio_event,0.010874,0.010811,0.000064
9,ETH,LSTM,LSTM-hs32,exceedance_down,0.011027,0.010811,0.000216



Coin: BTC | Metric: QLIKE | Rows in delta table: 21


,coin,model_kind,model_name,combo_label,qlike,baseline_loss,loss_delta_vs_baseline
0,BTC,HAR,HAR-baseline,None,0.333125,0.333125,0.000000
1,BTC,HAR,HAR-baseline,ratio_event,0.327857,0.333125,-0.005268
2,BTC,HAR,HAR-baseline,exceedance_down,0.321976,0.333125,-0.011149
3,BTC,HAR,HAR-baseline,lambda_t,0.326461,0.333125,-0.006664
4,BTC,HAR,HAR-baseline,"[ratio_event, exceedance_down]",0.316842,0.333125,-0.016283
5,BTC,HAR,HAR-baseline,"[exceedance_down, lambda_t]",0.317603,0.333125,-0.015522
6,BTC,HAR,HAR-baseline,"[ratio_event, exceedance_down, lambda_t]",0.315066,0.333125,-0.018058
7,BTC,LSTM,LSTM-hs32,None,0.430084,0.430084,0.000000
8,BTC,LSTM,LSTM-hs32,ratio_event,0.443125,0.430084,0.013041
9,BTC,LSTM,LSTM-hs32,exceedance_down,0.429035,0.430084,-0.001050



Coin: ETH | Metric: QLIKE | Rows in delta table: 21


,coin,model_kind,model_name,combo_label,qlike,baseline_loss,loss_delta_vs_baseline
0,ETH,HAR,HAR-baseline,None,0.288198,0.288198,0.000000
1,ETH,HAR,HAR-baseline,ratio_event,0.284042,0.288198,-0.004155
2,ETH,HAR,HAR-baseline,exceedance_down,0.284324,0.288198,-0.003873
3,ETH,HAR,HAR-baseline,lambda_t,0.289686,0.288198,0.001489
4,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down]",0.281678,0.288198,-0.006520
5,ETH,HAR,HAR-baseline,"[exceedance_down, lambda_t]",0.288605,0.288198,0.000407
6,ETH,HAR,HAR-baseline,"[ratio_event, exceedance_down, lambda_t]",0.286454,0.288198,-0.001744
7,ETH,LSTM,LSTM-hs32,None,0.411943,0.411943,0.000000
8,ETH,LSTM,LSTM-hs32,ratio_event,0.401369,0.411943,-0.010575
9,ETH,LSTM,LSTM-hs32,exceedance_down,0.570539,0.411943,0.158596


## 3) Mean and Std of Loss Across Feature Variations

For each metric, this section:
- computes the mean loss across all selected feature combinations
- computes the standard deviation of that loss
- displays a table ordered by increasing mean loss
- plots a mean-loss comparison chart with std as error bars

Lower mean loss is better.

In [50]:
metrics_for_summary = [m for m in METRICS_FOR_DELTA if m in summary_df.columns]

if not metrics_for_summary:
    raise ValueError("No metrics available for the mean/std summary section")

for metric in metrics_for_summary:
    for coin in sorted(summary_df["coin"].dropna().astype(str).unique().tolist()):
        coin_df = summary_df.loc[summary_df["coin"].astype(str) == coin].copy()

        mean_std_df = (
            coin_df.groupby(["model_kind", "model_name"], as_index=False)[metric]
            .agg(mean_loss="mean", std_loss="std", n_feature_variations="count")
            .sort_values(["mean_loss", "std_loss", "model_name"], ascending=[True, True, True])
            .reset_index(drop=True)
        )
        mean_std_df["coin"] = coin
        mean_std_df["std_loss"] = mean_std_df["std_loss"].fillna(0.0)
        mean_std_df = mean_std_df[["coin", "model_kind", "model_name", "mean_loss", "std_loss", "n_feature_variations"]]

        print(f"\nCoin: {coin} | Metric: {metric.upper()} | Mean and std across feature variations")
        display(mean_std_df)

        plot_df = mean_std_df.sort_values(["mean_loss", "std_loss", "model_name"]).copy()
        plot_df["model_name"] = pd.Categorical(
            plot_df["model_name"],
            categories=plot_df["model_name"].tolist(),
            ordered=True,
        )

        fig = px.bar(
            plot_df,
            x="mean_loss",
            y="model_name",
            color="model_kind",
            error_x="std_loss",
            orientation="h",
            title=f"{coin} | Mean loss across feature variations ({metric.upper()})",
            labels={
                "mean_loss": f"Mean {metric.upper()}",
                "std_loss": f"Std {metric.upper()}",
                "model_name": "Model variant",
                "model_kind": "Family",
            },
        )
        fig.update_layout(template="plotly_white")
        fig.show()


Coin: BTC | Metric: MSE | Mean and std across feature variations


,coin,model_kind,model_name,mean_loss,std_loss,n_feature_variations
0,BTC,HAR,HAR-baseline,0.000109,0.000001,7
1,BTC,LSTM,LSTM-hs32,0.000150,0.000007,7
2,BTC,LSTM,LSTM-hs64,0.000152,0.000008,7



Coin: ETH | Metric: MSE | Mean and std across feature variations


,coin,model_kind,model_name,mean_loss,std_loss,n_feature_variations
0,ETH,HAR,HAR-baseline,0.000165,0.000002,7
1,ETH,LSTM,LSTM-hs64,0.000236,0.000007,7
2,ETH,LSTM,LSTM-hs32,0.000239,0.000006,7



Coin: BTC | Metric: MAE | Mean and std across feature variations


...

,coin,model_kind,model_name,mean_loss,std_loss,n_feature_variations
0,ETH,HAR,HAR-baseline,0.008831,0.000095,7
1,ETH,LSTM,LSTM-hs64,0.010829,0.000298,7
2,ETH,LSTM,LSTM-hs32,0.010859,0.000092,7



Coin: BTC | Metric: QLIKE | Mean and std across feature variations


,coin,model_kind,model_name,mean_loss,std_loss,n_feature_variations
0,BTC,HAR,HAR-baseline,0.322704,0.006690,7
1,BTC,LSTM,LSTM-hs32,0.435647,0.009538,7
2,BTC,LSTM,LSTM-hs64,0.446435,0.011247,7



Coin: ETH | Metric: QLIKE | Mean and std across feature variations


,coin,model_kind,model_name,mean_loss,std_loss,n_feature_variations
0,ETH,HAR,HAR-baseline,0.286141,0.002904,7
1,ETH,LSTM,LSTM-hs64,0.416055,0.015938,7
2,ETH,LSTM,LSTM-hs32,0.437391,0.059318,7


## 4) QLIKE Summary Tables in LaTeX Format

For each coin, this section builds a LaTeX-ready table with:
- rows: model variants
- columns: feature combinations
- values: `loss_delta_vs_baseline` using `QLIKE`

Formatting rule:
- negative values: `\green{...}`
- positive values: `\red{...}`
- zero values: plain text

These tables are ready to copy and paste into LaTeX with your predefined `\green` and `\red` commands.

In [51]:
QLIKE_METRIC = "qlike"

if QLIKE_METRIC not in summary_df.columns:
    raise ValueError("QLIKE metric is not available in summary_df")


def format_delta_for_latex(value: float) -> str:
    if pd.isna(value):
        return ""
    formatted = f"{value:.4f}"
    if value < 0:
        return f"\\color{{green}}{{{formatted}}}"
    if value > 0:
        return f"\\color{{red}}{{{formatted}}}"
    return formatted


def escape_latex_underscores(value: str) -> str:
    return str(value).replace("_", "\\_")


for coin in sorted(summary_df["coin"].dropna().astype(str).unique().tolist()):
    coin_df = summary_df.loc[summary_df["coin"].astype(str) == coin].copy()

    agg = (
        coin_df.groupby(["model_name", "combo_label"], as_index=False)[QLIKE_METRIC]
        .mean(numeric_only=True)
    )

    baseline = (
        agg.loc[agg["combo_label"] == "None", ["model_name", QLIKE_METRIC]]
        .rename(columns={QLIKE_METRIC: "baseline_loss"})
    )

    delta_df = agg.merge(baseline, on="model_name", how="left")
    delta_df = delta_df.dropna(subset=["baseline_loss"]).copy()
    delta_df["loss_delta_vs_baseline"] = delta_df[QLIKE_METRIC] - delta_df["baseline_loss"]

    delta_pivot = delta_df.pivot_table(
        index="model_name",
        columns="combo_label",
        values="loss_delta_vs_baseline",
        aggfunc="mean",
    )
    delta_pivot = delta_pivot.reindex(columns=COMBO_ORDER)

    non_baseline_cols = [col for col in COMBO_ORDER if col != "None"]
    row_order = (
        delta_pivot[non_baseline_cols]
        .mean(axis=1, skipna=True)
        .sort_values(ascending=True)
        .index
    )
    delta_pivot = delta_pivot.reindex(row_order)

    formatted_table = delta_pivot.applymap(format_delta_for_latex)
    formatted_table.index = [escape_latex_underscores(name) for name in formatted_table.index]
    formatted_table.columns = [escape_latex_underscores(name) for name in formatted_table.columns]
    formatted_table.index.name = "Model"
    formatted_table.columns.name = "Features"

    print(f"\nCoin: {coin} | QLIKE delta vs baseline (LaTeX-ready table)")
    display(formatted_table)

    latex_table = formatted_table.to_latex(escape=False)
    print("LaTeX output:")
    print(latex_table)


Coin: BTC | QLIKE delta vs baseline (LaTeX-ready table)


C:\Users\diego\AppData\Local\Temp\ipykernel_10392\2167246879.py:56: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



Features,None,ratio\_event,exceedance\_down,lambda\_t,"[ratio\_event, exceedance\_down]","[exceedance\_down, lambda\_t]","[ratio\_event, exceedance\_down, lambda\_t]"
Model,,,,,,,
HAR-baseline,0.0000,\color{green}{-0.0053},\color{green}{-0.0111},\color{green}{-0.0067},\color{green}{-0.0163},\color{green}{-0.0155},\color{green}{-0.0181}
LSTM-hs32,0.0000,\color{red}{0.0130},\color{green}{-0.0010},\color{green}{-0.0029},\color{red}{0.0028},\color{red}{0.0034},\color{red}{0.0237}
LSTM-hs64,0.0000,\color{red}{0.0126},\color{red}{0.0086},\color{red}{0.0156},\color{red}{0.0201},\color{red}{0.0365},\color{red}{0.0143}


LaTeX output:
\begin{tabular}{llllllll}
\toprule
Features & None & ratio\_event & exceedance\_down & lambda\_t & [ratio\_event, exceedance\_down] & [exceedance\_down, lambda\_t] & [ratio\_event, exceedance\_down, lambda\_t] \\
Model &  &  &  &  &  &  &  \\
\midrule
HAR-baseline & 0.0000 & \color{green}{-0.0053} & \color{green}{-0.0111} & \color{green}{-0.0067} & \color{green}{-0.0163} & \color{green}{-0.0155} & \color{green}{-0.0181} \\
LSTM-hs32 & 0.0000 & \color{red}{0.0130} & \color{green}{-0.0010} & \color{green}{-0.0029} & \color{red}{0.0028} & \color{red}{0.0034} & \color{red}{0.0237} \\
LSTM-hs64 & 0.0000 & \color{red}{0.0126} & \color{red}{0.0086} & \color{red}{0.0156} & \color{red}{0.0201} & \color{red}{0.0365} & \color{red}{0.0143} \\
\bottomrule
\end{tabular}


Coin: ETH | QLIKE delta vs baseline (LaTeX-ready table)


C:\Users\diego\AppData\Local\Temp\ipykernel_10392\2167246879.py:56: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



Features,None,ratio\_event,exceedance\_down,lambda\_t,"[ratio\_event, exceedance\_down]","[exceedance\_down, lambda\_t]","[ratio\_event, exceedance\_down, lambda\_t]"
Model,,,,,,,
HAR-baseline,0.0000,\color{green}{-0.0042},\color{green}{-0.0039},\color{red}{0.0015},\color{green}{-0.0065},\color{red}{0.0004},\color{green}{-0.0017}
LSTM-hs64,0.0000,\color{red}{0.0044},\color{red}{0.0243},\color{green}{-0.0079},\color{red}{0.0236},\color{red}{0.0120},\color{red}{0.0376}
LSTM-hs32,0.0000,\color{green}{-0.0106},\color{red}{0.1586},\color{red}{0.0015},\color{red}{0.0042},\color{red}{0.0178},\color{red}{0.0067}


LaTeX output:
\begin{tabular}{llllllll}
\toprule
Features & None & ratio\_event & exceedance\_down & lambda\_t & [ratio\_event, exceedance\_down] & [exceedance\_down, lambda\_t] & [ratio\_event, exceedance\_down, lambda\_t] \\
Model &  &  &  &  &  &  &  \\
\midrule
HAR-baseline & 0.0000 & \color{green}{-0.0042} & \color{green}{-0.0039} & \color{red}{0.0015} & \color{green}{-0.0065} & \color{red}{0.0004} & \color{green}{-0.0017} \\
LSTM-hs64 & 0.0000 & \color{red}{0.0044} & \color{red}{0.0243} & \color{green}{-0.0079} & \color{red}{0.0236} & \color{red}{0.0120} & \color{red}{0.0376} \\
LSTM-hs32 & 0.0000 & \color{green}{-0.0106} & \color{red}{0.1586} & \color{red}{0.0015} & \color{red}{0.0042} & \color{red}{0.0178} & \color{red}{0.0067} \\
\bottomrule
\end{tabular}



## 5) Correlation Heatmaps with $\sqrt{RV}$

For each coin, this section builds a Pearson-correlation heatmap using the available daily variables and $\sqrt{RV}$.

Interpretation:
- values close to 1 indicate strong positive linear association
- values close to -1 indicate strong negative linear association
- values near 0 indicate weak linear association

In [52]:
CORRELATION_FEATURES = [feature for feature in FEATURE_COLUMNS if feature != "RV"]

for coin in sorted(features_df["coin"].dropna().astype(str).unique().tolist()):
    coin_df = features_df.loc[features_df["coin"].astype(str) == coin].copy()

    available_corr_features = [feature for feature in CORRELATION_FEATURES if feature in coin_df.columns]
    if not available_corr_features:
        print(f"Skipping {coin}: no correlation features available")
        continue

    corr_input = coin_df[["date"] + available_corr_features].copy()
    corr_input["RV"] = pd.to_numeric(coin_df.get("RV"), errors="coerce")
    corr_input["sqrt_RV"] = np.where(corr_input["RV"] >= 0, np.sqrt(corr_input["RV"]), np.nan)

    corr_cols = available_corr_features + ["sqrt_RV"]
    corr_input = corr_input[corr_cols].apply(pd.to_numeric, errors="coerce")
    corr_input = corr_input.dropna(how="any").copy()

    if corr_input.empty:
        print(f"Skipping {coin}: insufficient complete observations for correlation heatmap")
        continue

    corr_mat = corr_input.corr(method="pearson")

    fig = px.imshow(
        corr_mat,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title=f"{coin} | Pearson correlation heatmap of daily variables and sqrt(RV)",
        labels={"x": "Variable", "y": "Variable", "color": "Correlation"},
    )
    fig.update_layout(template="plotly_white")
    fig.show()

    print(f"Correlation matrix for {coin}")
    display(corr_mat.round(4))

Correlation matrix for BTC


,ratio_event,exceedance_down,lambda_t,sqrt_RV
ratio_event,1.0000,0.5722,0.0922,0.4467
exceedance_down,0.5722,1.0000,-0.0553,0.7958
lambda_t,0.0922,-0.0553,1.0000,-0.1965
sqrt_RV,0.4467,0.7958,-0.1965,1.0000


Correlation matrix for ETH


,ratio_event,exceedance_down,lambda_t,sqrt_RV
ratio_event,1.0000,0.6083,0.0820,0.4817
exceedance_down,0.6083,1.0000,0.0009,0.7948
lambda_t,0.0820,0.0009,1.0000,-0.0712
sqrt_RV,0.4817,0.7948,-0.0712,1.0000
